1. Database Connection

In [106]:
import os
import sys
from dotenv import load_dotenv

# Load environment params
load_dotenv()
un = os.getenv("user")
pw = os.getenv("password")
cs = os.getenv("connstr")
#print("un: " + un + "\npw: " + pw + "\ncs: " + cs)

# Connect to the database
import oracledb

# Connection to Oracle database
def dbConnection():
    """데이터베이스 연결을 시도하고 연결 객체를 반환합니다."""    
    try:
        connection = oracledb.connect(user=un, password=pw, dsn=cs)        
        
        # Check Databse Vesion
        db_version = tuple(int(s) for s in connection.version.split("."))
        if db_version < (23, 4):
            # Close connection
            connection.close()
            # Print error message
            print("\nConnection to Oracle Database failed")
            sys.exit("Requires Oracle Databse 23.4 or later.")
        else:
            print("\nConnection to Oracle Database")
            print("Oracle Database " + connection.version)        

        # Return database connection object
        return connection

    except oracledb.Error as e:
        error, = e.args
        print(f"\nConnection to Oracle Database failed : {error.code} - {error.message}")
        return None
        
    except Exception as e:                
        print("Raise unexcpeted error: " + str(e))
        return None

# Check connection Oracle database
def ensureDbConnection(connection):
    """현재 연결 상태를 확인하고, 필요한 경우 재연결을 수행합니다."""
    if connection:
        try:
            with connection.cursor() as cursor:
                cursor.execute("SELECT 1 FROM dual")

            return connection
        except oracledb.Error:
            print("기존 연결이 유효하지 않습니다. 재연결을 시도합니다.")
            connection.close()
    else:
        print("연결 객체가 없습니다. 새 연결을 시도합니다.")

    #재연결 시도
    for attemp in range(5):
        print(f"재연결 시도 중... (시도 횟수: {attempt + 1})")
        new_connection = dbConnection()
        
        if new_connection:
            return new_connection
        
        time.sleep(2) # 2초 대기 후 재시도
    
    raise Exception("데이터베이스 재연결에 실패했습니다.")

2. Datase Close

In [76]:
def dbClose():
    if connection:
        if connection.is_healthy():
            connection.cursor().close()
            connection.close()
            print("\nClosed connection to Oracle Database")
        else:
            print("\nAlready closed connection to Oracle Database")

3. Test

In [111]:
# 초기 연결
connection = dbConnection()


##########################################
# 애플리케이션 로직에서 연결 사용 시
##########################################
try:
    connection = ensureDbConnection(connection)
    with connection.cursor() as cursor:
        cursor.execute("SELECT sysdate FROM DUAL")
        for row in cursor:
            print(f"\n현재 시간: {row[0]}")

except Exception as e:
    print(f"오류 발생: {e}")
finally:
    dbClose()
    #if connection:
    #    connection.close() # 작업 완료 후 연결 종료


Connection to Oracle Database
Oracle Database 23.9.0.25.7

현재 시간: 2025-11-07 18:06:24

Closed connection to Oracle Database
